# Single-Series Insert and Read

The core workflow: create a series identified by name and labels, insert a DataFrame, and read it back. Series are scoped with `.get_series().where()` — a chainable filter that returns a `SeriesCollection` pointing at one or more matching series.

In [1]:
try:
    import google.colab
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py',
        '/tmp/colab_setup.py'
    )
    exec(open('/tmp/colab_setup.py').read())
except ImportError:
    pass

In [2]:
from datetime import datetime, timezone, timedelta
import pandas as pd
import polars as pl
import timedb as td

base_time = datetime(2025, 1, 1, tzinfo=timezone.utc)

## Setup

Two `wind_power` series at different sites. Name + labels together form the identity — `site="Gotland"` and `site="Skane"` are distinct series even though they share the same name.

In [3]:
td.delete()
td.create()

for site in ["Gotland", "Skane"]:
    sid = td.create_series("wind_power", unit="MW", labels={"site": site})
    print(f"✓ wind_power site={site}: series_id={sid}")

Creating database schema...
  ✓ PostgreSQL: series_table
  ✓ ClickHouse: runs_table, flat, overlapping_short/medium/long
✓ Schema created successfully
✓ wind_power site=Gotland: series_id=1


✓ wind_power site=Skane: series_id=2


## Insert

Both pandas and polars DataFrames are accepted. Columns must be `[valid_time, value]` with a timezone-aware datetime.

In [4]:
df_polars = pl.DataFrame({
    "valid_time": pl.Series([base_time + timedelta(hours=i) for i in range(24)]).cast(pl.Datetime("us", "UTC")),
    "value": [5.0 + i * 0.3 for i in range(24)],
})

result = td.get_series("wind_power").where(site="Gotland").insert(df_polars)
print(f"✓ Gotland (polars): {result}")

✓ Gotland (polars): InsertResult(run_id=UUID('019d2dde-6e51-7169-ac40-648bc766f7e7'), workflow_id='sdk-workflow', series_id=1)


In [5]:
df_pandas = pd.DataFrame({
    "valid_time": pd.date_range(base_time, periods=24, freq="h", tz="UTC"),
    "value": [4.0 + i * 0.25 for i in range(24)],
})

result = td.get_series("wind_power").where(site="Skane").insert(df_pandas)
print(f"✓ Skane (pandas):  {result}")

✓ Skane (pandas):  InsertResult(run_id=UUID('019d2dde-6e75-72f0-9fef-580863b38517'), workflow_id='sdk-workflow', series_id=2)


## Label Routing

`.where()` filters to a specific series. Chain multiple filters — each call returns a new `SeriesCollection`.

In [6]:
all_wind = td.get_series("wind_power")
gotland  = all_wind.where(site="Gotland")
print(f"All wind_power: {all_wind.count()} series")
print(f"site=Gotland:   {gotland.count()} series")
for s in all_wind.list_series():
    print(f"  series_id={s['series_id']}: labels={s['labels']}")

All wind_power: 2 series
site=Gotland:   1 series
  series_id=1: labels={'site': 'Gotland'}
  series_id=2: labels={'site': 'Skane'}


## Read

`read()` returns a `TimeSeries` object. Call `.to_polars()` or `.to_pandas()` to get a DataFrame.

In [7]:
ts = td.get_series("wind_power").where(site="Gotland").read()
print(ts)

TimeSeries
┌─────────────────────────────────────────┐
│  Name:             wind_power           │
│  Shape:            SIMPLE               │
│  Rows:             24                   │
│  Timezone:         UTC                  │
│  Unit:             MW                   │
│  Labels:           {'site': 'Gotland'}  │
├─────────────────────────────────────────┤
│                    wind_power           │
│  2025-01-01 00:00         5.0           │
│  2025-01-01 01:00         5.3           │
│  2025-01-01 02:00         5.6           │
│  ...                      ...           │
│  2025-01-01 21:00        11.3           │
│  2025-01-01 22:00        11.6           │
│  2025-01-01 23:00        11.9           │
└─────────────────────────────────────────┘


## Multiple Series

Read each series and combine into a single DataFrame for comparison.

In [8]:
dfs = {}
for s in td.get_series("wind_power").list_series():
    site = s["labels"]["site"]
    ts = td.get_series("wind_power").where(site=site).read()
    dfs[site] = ts.to_polars().rename({"value": site})

combined = dfs["Gotland"].join(dfs["Skane"], on="valid_time")
print(combined.head())

shape: (5, 3)
┌─────────────────────────┬─────────┬───────┐
│ valid_time              ┆ Gotland ┆ Skane │
│ ---                     ┆ ---     ┆ ---   │
│ datetime[μs, UTC]       ┆ f64     ┆ f64   │
╞═════════════════════════╪═════════╪═══════╡
│ 2025-01-01 00:00:00 UTC ┆ 5.0     ┆ 4.0   │
│ 2025-01-01 01:00:00 UTC ┆ 5.3     ┆ 4.25  │
│ 2025-01-01 02:00:00 UTC ┆ 5.6     ┆ 4.5   │
│ 2025-01-01 03:00:00 UTC ┆ 5.9     ┆ 4.75  │
│ 2025-01-01 04:00:00 UTC ┆ 6.2     ┆ 5.0   │
└─────────────────────────┴─────────┴───────┘
